In [ ]:
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

# TODO: Update these variables when running notebook
default_match_file = "Shot_Visuals_CassiusChinlund_GregGamal.csv"
if os.path.exists(os.path.join("..", "match-csvs", default_match_file)):
    path = os.path.join("..", "match-csvs", default_match_file)
else:
    path = os.path.join("data", "match-csvs", default_match_file)

# remember to change the player name to the current matches' UCLA player name
ucla_player = "Cassius Chinlund"
output_filename = "avg_forehand_depth.json"

# You can pass one match path or multiple paths.
# Example for multiple:
# paths = [
#     "../match-csvs/Shot_Visuals_RudyQuan_IiroVasa.csv",
#     "../match-csvs/Shot_Visuals_RudyQuan_AristotelisThanos.csv"
# ]
paths = [path]

# Adjusted output path for JSONs (works whether cwd is repo root or data/notebooks)
if os.path.isdir("data/json"):
    output_dir = os.path.join(os.getcwd(), "data/json")
elif os.path.isdir("../json"):
    output_dir = os.path.join(os.getcwd(), "../json")
else:
    output_dir = os.path.join(os.getcwd(), "data", "json")
os.makedirs(output_dir, exist_ok=True)


def _clean_value(value, fallback="Unknown"):
    if pd.isna(value):
        return fallback
    text = str(value).strip()
    if text == "" or text.lower() == "nan":
        return fallback
    return text


def _resolve_match_path(match_path):
    if os.path.exists(match_path):
        return match_path

    file_name = os.path.basename(match_path)
    candidates = [
        os.path.join("data", "match-csvs", file_name),
        os.path.join("..", "match-csvs", file_name)
    ]

    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate

    raise FileNotFoundError(f"Could not find match CSV: {match_path}")


def _build_match_record(events, player, source_path):
    required_cols = {'shotHitBy', 'shotFhBh', 'shotLocationY'}
    missing = [c for c in required_cols if c not in events.columns]
    if missing:
        raise ValueError(f"Missing required columns in {source_path}: {missing}")

    forehands = events[
        (events['shotHitBy'] == player) &
        (events['shotFhBh'].astype(str).str.lower() == 'forehand')
    ].copy()

    forehands['depth'] = pd.to_numeric(forehands['shotLocationY'], errors='coerce').abs()
    forehands = forehands[forehands['depth'].notna()]

    if forehands.empty:
        return None

    first_row = events.iloc[0]

    player1 = _clean_value(first_row.get('player1Name'), fallback='')
    player2 = _clean_value(first_row.get('player2Name'), fallback='Unknown Opponent')
    if player1 == player and player2 != 'Unknown Opponent':
        opponent = player2
    elif player2 == player and player1 != '':
        opponent = player1
    else:
        opponent = player2 if player2 != '' else _clean_value(first_row.get('opponentTeam'), 'Unknown Opponent')

    depth_series = forehands['depth']

    return {
        'matchId': f"vs {opponent}",
        'sourceFile': os.path.basename(source_path),
        'opponent': opponent,
        'date': _clean_value(first_row.get('Date')),
        'round': _clean_value(first_row.get('Round')),
        'surface': _clean_value(first_row.get('Surface')),
        'forehandCount': int(len(depth_series)),
        'averageDepth': round(float(depth_series.mean()), 1),
        'medianDepth': round(float(depth_series.median()), 1),
        'minDepth': round(float(depth_series.min()), 1),
        'maxDepth': round(float(depth_series.max()), 1),
        'depthValues': [round(float(x), 1) for x in depth_series.tolist()]
    }


def average_forehand_depth(player, match_paths, output_name="avg_forehand_depth.json"):
    records = []

    for match_path in match_paths:
        resolved_path = _resolve_match_path(match_path)
        events = pd.read_csv(resolved_path)
        record = _build_match_record(events, player, resolved_path)
        if record is not None:
            records.append(record)

    if not records:
        raise ValueError("No valid forehand depth data found for the selected player/match paths.")

    all_depths = [value for record in records for value in record['depthValues']]
    all_depths_series = pd.Series(all_depths, dtype='float64')

    payload = {
        'player': player,
        'metricDefinition': 'Depth is calculated as abs(shotLocationY), where higher values indicate deeper forehands.',
        'matches': records,
        'overall': {
            'matchCount': int(len(records)),
            'totalForehands': int(len(all_depths)),
            'averageDepth': round(float(all_depths_series.mean()), 1),
            'medianDepth': round(float(all_depths_series.median()), 1),
            'q1': round(float(all_depths_series.quantile(0.25)), 1),
            'q3': round(float(all_depths_series.quantile(0.75)), 1),
            'iqr': round(float(all_depths_series.quantile(0.75) - all_depths_series.quantile(0.25)), 1)
        }
    }

    out_path = os.path.join(output_dir, output_name)
    with open(out_path, 'w') as f:
        json.dump(payload, f, separators=(',', ':'))

    print(f"Wrote {out_path}")
    print(f"Player: {player}")
    print(f"Matches: {len(records)} | Forehands: {len(all_depths)}")


# run the pipeline
average_forehand_depth(ucla_player, paths, output_filename)


Wrote /Users/pr3s10/Documents/GitHub/UCLA-Tennis-Viz/data/notebooks/../json/avg_forehand_depth.json
Player: Cassius Chinlund
Matches: 1 | Forehands: 205
